# Model Training: SVD & k-NN

## Training Pipeline
1. Load preprocessed data (trainset, testset)
2. GridSearchCV for SVD hyperparameters
3. GridSearchCV for k-NN (item-based) hyperparameters
4. Train best models on full trainset
5. Evaluate on testset
6. Save models to `data/processed/models/`

In [ ]:
import sys
sys.path.append('..')

from src.recommender import (
    train_svd,
    train_knn,
    evaluate_model,
    train_and_save_models,
    save_model,
    load_model,
    get_top_n_recommendations,
    get_popularity_recommendations,
)
from src.preprocessing import load_preprocessed_data

data = load_preprocessed_data()
train_dataset = data["train_data"]
testset = data["test_data"].build_full_trainset().build_testset()
trainset = train_dataset.build_full_trainset()

print(f"Trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")
print(f"Testset: {len(testset)} ratings")

In [ ]:
# Train and save both models
results = train_and_save_models(train_dataset, testset, cv=3, n_jobs=-1)

In [ ]:
# Load saved models and test recommendations
svd_algo = load_model("algo_svd")
knn_algo = load_model("algo_knn")

# Test recommendations for sample users
for uid in [1, 10, 100, 500]:
    svd_recs = get_top_n_recommendations(svd_algo, uid, trainset, n=5)
    knn_recs = get_top_n_recommendations(knn_algo, uid, trainset, n=5)
    pop_recs = get_popularity_recommendations(trainset, n=5)
    print(f"User {uid}:")
    print(f"  SVD: {[(iid, round(float(est), 2)) for iid, est in svd_recs]}")
    print(f"  k-NN: {[(iid, round(float(est), 2)) for iid, est in knn_recs]}")
    print(f"  Pop: {[(iid, round(float(avg), 2)) for iid, avg in pop_recs]}")

In [ ]:
# Load grid search results for analysis
import joblib
from pathlib import Path

MODELS_DIR = Path("../data/processed/models")
svd_gs = joblib.load(MODELS_DIR / "svd_gridsearch.pkl")
knn_gs = joblib.load(MODELS_DIR / "knn_gridsearch.pkl")

print("SVD Grid Search Results:")
print(f"  Best RMSE: {svd_gs.best_score['rmse']:.4f}")
print(f"  Best MAE: {svd_gs.best_score['mae']:.4f}")
print(f"  Best params: {svd_gs.best_params['rmse']}")

print("\nk-NN Grid Search Results:")
print(f"  Best RMSE: {knn_gs.best_score['rmse']:.4f}")
print(f"  Best MAE: {knn_gs.best_score['mae']:.4f}")
print(f"  Best params: {knn_gs.best_params['rmse']}")

In [ ]:
# Evaluation with comprehensive metrics
from src.evaluation import compare_models, popularity_baseline_recommendations, evaluate_recommender

models = {"SVD": svd_algo, "k-NN": knn_algo}
results = compare_models(models, testset, trainset, k=10, threshold=4.0)
print(results.to_string())

print("\nPopularity Baseline:")
pop_predictions = popularity_baseline_recommendations(trainset, testset, k=10)
class PopBaseline:
    def test(self, ts):
        return pop_predictions
pop_metrics = evaluate_recommender(PopBaseline(), testset, trainset, k=10, threshold=4.0)
for k, v in pop_metrics.items():
    print(f"  {k}: {float(v):.4f}")

## Expected Results

| Metric | SVD Target | k-NN Target | Popularity Baseline |
|--------|------------|-------------|---------------------|
| RMSE | < 0.90 | < 0.95 | ~1.33 |
| MAE | < 0.70 | < 0.75 | ~1.07 |
| Precision@10 | > 0.15 | > 0.15 | ~0.07 |
| Recall@10 | > 0.10 | > 0.10 | ~0.63 |
| Coverage@10 | > 0.30 | > 0.30 | ~0.003 |

SVD should outperform k-NN on RMSE/MAE. Both should significantly beat popularity baseline on Precision@10.